# Web Scraping Day 2: When Websites Fight Back — JavaScript & Selenium

## Week 6, Day 2 — Scraping Dynamic Pages

---

### Today's Journey (~2 Hours)

| Part | Topic | Duration |
|------|-------|----------|
| 1 | The Problem — Why `requests` Isn't Always Enough | ~15 min |
| 2 | JavaScript 101 — Just Enough to Understand the Enemy | ~20 min |
| | **BREAK 1** | 10 min |
| 3 | Meet Selenium — Automating a Real Browser | ~20 min |
| 4 | Selenium + BeautifulSoup — The Power Combo | ~20 min |
| | **BREAK 2** | 10 min |
| 5 | Exercises — Scraping Dynamic Websites | ~25 min |
| 6 | Daily Challenge — Scraping a New JS Site | ~10 min |

---

```
TODAY'S PROMISE:

    ┌─────────────────────────────────────────────────────────────────────┐
    │                                                                     │
    │   BEFORE (today):                                                  │
    │   "I can scrape static HTML pages with requests + BeautifulSoup"   │
    │                                                                     │
    │   AFTER (today):                                                   │
    │   "I can scrape ANY page — even JavaScript-powered ones"           │
    │                                                                     │
    │   Static HTML → requests + BS4     (Day 1)                         │
    │   Dynamic JS  → Selenium + BS4     (Day 2)  ← TODAY                │
    │                                                                     │
    └─────────────────────────────────────────────────────────────────────┘
```

In [ ]:
# Install libraries (google-colab-selenium handles Chrome + driver automatically)
!pip install -q google-colab-selenium beautifulsoup4
print("\u2705 Libraries installed!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 17.4 MB/s eta 0:00:00
✅ Libraries installed!


In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import google_colab_selenium as gs
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
print("Ready for Web Scraping Day 2!")

Ready for Web Scraping Day 2!


---

# Part 1: The Problem — Why `requests` Isn't Always Enough

---

### The Same Website, Two Versions — One Works, One Doesn't

In [ ]:
# The STATIC version of the quotes site — Day 1 skills work perfectly
url_static = "https://quotes.toscrape.com/"
soup_static = BeautifulSoup(requests.get(url_static).text, 'html.parser')
quotes_static = soup_static.find_all(class_='quote')
print(f"=== STATIC version ({url_static}) ===")
print(f"Quotes found: {len(quotes_static)}")
first = quotes_static[0]
print(f'First quote: {first.find(class_="text").get_text()[:60]}...')
print(f'First author: {first.find(class_="author").get_text()}')

print()

# The JAVASCRIPT version of the same site — Day 1 skills FAIL
url_js = "https://quotes.toscrape.com/js/"
soup_js = BeautifulSoup(requests.get(url_js).text, 'html.parser')
quotes_js = soup_js.find_all(class_='quote')
print(f"=== JAVASCRIPT version ({url_js}) ===")
print(f"Quotes found: {len(quotes_js)}")
print(f"Contains 'Einstein'? {'Einstein' in soup_js.get_text()}")

# Where IS the data? Trapped in JavaScript code:
print()
print("But the data IS here — trapped in a <script> tag:")
scripts = [s for s in soup_js.find_all('script') if s.string and len(s.string) > 100]
if scripts:
    print(scripts[0].string[:200].strip())
    print("  ...")

=== STATIC version (https://quotes.toscrape.com/) ===
Quotes found: 10
First quote: “The world as we have created it is a process of our thinkin...
First author: Albert Einstein

=== JAVASCRIPT version (https://quotes.toscrape.com/js/) ===
Quotes found: 0
Contains 'Einstein'? False

But the data IS here — trapped in a <script> tag:
var data = [
    {
        "tags": [
            "change",
            "deep-thoughts",
            "thinking",
            "world"
        ],
        "author": {
            "name": "Albert Eins
  ...


### Why Did That Fail?

```
STATIC PAGE (Day 1):                    DYNAMIC PAGE (Day 2):
════════════════════                    ═════════════════════

Server sends:                           Server sends:
┌────────────────────┐                  ┌────────────────────┐
│ <h1>Title</h1>     │                  │ <script>           │
│ <p>Content here</p>│                  │   var data = [...]  │
│ <a href="...">     │                  │   // JS builds the │
│                    │                  │   // page content   │
└────────────────────┘                  └────────────────────┘
        │                                       │
        ▼                                       ▼
  requests.get()                          requests.get()
  sees ALL content ✓                      sees EMPTY page ✗
                                                │
                                          Need a real browser!
```

### How to Tell If a Page Uses JavaScript

| Method | How | Result |
|--------|-----|--------|
| **View Source vs Inspect** | Right-click → View Source shows raw HTML; Inspect shows after JS | If very different → JS page |
| **Disable JS** | DevTools → Settings → uncheck JavaScript → refresh | If page goes blank → JS page |
| **Just try requests** | If `find_all()` returns nothing on a page with visible content | → It's JavaScript |

---

# Part 2: JavaScript 101 — Just Enough to Understand the Enemy

---

### The Three Languages of the Web

| Language | Role | Analogy | `requests` | Selenium |
|----------|------|---------|-----------|----------|
| **HTML** | Structure | Skeleton | Reads ✓ | Reads ✓ |
| **CSS** | Style | Clothing | Reads ✓ | Reads ✓ |
| **JavaScript** | Behavior | Muscles | **Can't run ✗** | **Runs ✓** |

JavaScript builds content AFTER the page loads — infinite scrolling,
dynamic updates, personalized content. `requests` misses all of it.

In [ ]:
# Static HTML — content is right in the source
static_html = """<html><body>
  <p class="quote">"Do great work." — Steve Jobs</p>
  <p class="quote">"Be the change." — Gandhi</p>
</body></html>"""

# Dynamic HTML — content is built by JavaScript AFTER the page loads
dynamic_html = """<html><body>
  <div id="quotes"></div>
  <script>
    var data = [{text: "Do great work.", author: "Steve Jobs"}];
    document.getElementById("quotes").innerHTML =
      data.map(q => '<p class="quote">' + q.text + '</p>').join('');
  </script>
</body></html>"""

static_soup = BeautifulSoup(static_html, 'html.parser')
dynamic_soup = BeautifulSoup(dynamic_html, 'html.parser')

print(f"Static page  → quotes found: {len(static_soup.find_all(class_='quote'))}")
print(f"Dynamic page → quotes found: {len(dynamic_soup.find_all(class_='quote'))}")
print()
print("requests sees the script but can\'t run it:")
print(f"  {dynamic_soup.find('script').string.strip()[:80]}...")

Static page  → quotes found: 2
Dynamic page → quotes found: 0

requests sees the script but can't run it:
  var data = [{text: "Do great work.", author: "Steve Jobs"}];
    document.getEle...


### JavaScript Concepts That Matter for Scraping

| Concept | What It Does | Example |
|---------|-------------|---------|
| **DOM Manipulation** | JS modifies the HTML | `document.getElementById("x").innerHTML = "..."` |
| **Fetch / AJAX** | JS loads data from server | `fetch('/api/data').then(...)` |
| **Event Handlers** | JS reacts to clicks/scrolls | `button.addEventListener("click", fn)` |

```javascript
// This is what breaks requests-based scraping:
document.getElementById("content").innerHTML = "<p>New!</p>";
// requests sees: <div id="content"></div>           ← empty
// Browser sees:  <div id="content"><p>New!</p></div> ← filled
```

### Why Do Sites Use JavaScript?

| Reason | Example |
|--------|---------|
| **Speed** | YouTube loads layout first, videos after |
| **Interactivity** | Google Maps updates without page reload |
| **Single Page Apps** | Gmail, ChatGPT, Spotify never fully reload |
| **Personalization** | Netflix, TikTok show different content per user |

**Most websites in 2026 use JavaScript heavily.** That's why we need Selenium.

---

## BREAK 1 (10 minutes)

---

---

# Part 3: Meet Selenium — Automating a Real Browser

---

### What Is Selenium?

```
  Your Script            ChromeDriver              Chrome Browser
  ┌──────────────┐      ┌─────────────┐         ┌──────────────┐
  │ driver.get() │─────▶│ Translates   │────────▶│ Opens page   │
  │              │      │ Python →     │         │ Runs JS      │
  │ driver.      │◀─────│ Chrome       │◀────────│ Returns HTML │
  │ page_source  │      │ commands     │         │              │
  └──────────────┘      └─────────────┘         └──────────────┘
        │
        ▼
  ┌──────────────┐
  │ BeautifulSoup│ ── Same parsing as Day 1!
  └──────────────┘
```

In [ ]:
# Set up Selenium — google-colab-selenium handles Chrome, driver, and all options!
driver = gs.Chrome()
print("Selenium ready! Headless Chrome is running.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Selenium ready! Headless Chrome is running.


In [ ]:
# The page that DEFEATED requests... let's try with Selenium!
url = "https://quotes.toscrape.com/js/"

# Use driver.get(url) to load the page
driver.get(url)

# Parse with BeautifulSoup(driver.page_source, 'html.parser')
soup = BeautifulSoup(driver.page_source, 'html.parser')

# Find all quotes (class_='quote') and print text + author
quotes = soup.find_all(class_='quote')
print()
for i, quote in enumerate(quotes[:5]):
  text = quote.find(class_='text').get_text()
  author = quote.find(class_='author').get_text()
  print(f"  {i+1}. {text}")
  print(f"    - {author}")


  1. “The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”
    - Albert Einstein
  2. “It is our choices, Harry, that show what we truly are, far more than our abilities.”
    - J.K. Rowling
  3. “There are only two ways to live your life. One is as though nothing is a miracle. The other is as though everything is a miracle.”
    - Albert Einstein
  4. “The person, be it gentleman or lady, who has not pleasure in a good novel, must be intolerably stupid.”
    - Jane Austen
  5. “Imperfection is beauty, madness is genius and it's better to be absolutely ridiculous than absolutely boring.”
    - Marilyn Monroe


### requests vs Selenium — When to Use What

| | `requests` (Day 1) | Selenium (Day 2) |
|---|---|---|
| **Runs JavaScript** | No | Yes |
| **Speed** | Fast | Slower (real browser) |
| **Code** | `requests.get(url)` | `driver.get(url)` |
| **HTML** | `response.text` | `driver.page_source` |
| **Use when** | Static HTML | JS-powered pages |

**Rule:** Try `requests` first. Empty results? → Switch to Selenium.

---

# Part 4: Selenium + BeautifulSoup — The Power Combo

---

### WebDriverWait — Why and How

JavaScript doesn't always finish instantly. If you grab `driver.page_source` too early,
you get an empty page. `WebDriverWait` checks every half-second and continues
**the moment your element appears**.

```python
# The pattern — use this every time after driver.get():
wait = WebDriverWait(driver, 10)  # wait up to 10 seconds
wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, '.quote')))
# NOW parse with BeautifulSoup
```

| `By` Selector | Example |
|---------------|---------|
| `By.CSS_SELECTOR` | `'.quote'`, `'#main'`, `'div.content > p'` |
| `By.ID` | `'main-content'` |
| `By.CLASS_NAME` | `'quote'` |

In [ ]:
# Why WebDriverWait matters — two approaches compared
url = "https://quotes.toscrape.com/js/"

# Approach 1: Grab page immediately
driver.get(url)
soup1 = BeautifulSoup(driver.page_source, 'html.parser')
print(f"Without waiting: {len(soup1.find_all(class_='quote'))} quotes")

# Approach 2: Wait for .quote elements first, THEN parse
driver.get(url)
wait = WebDriverWait(driver, 10)
wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, '.quote')))
soup2 = BeautifulSoup(driver.page_source, 'html.parser')

print(f"With WebDriverWait: {len(soup2.find_all(class_='quote'))} quotes")
print()

Without waiting: 10 quotes
With WebDriverWait: 10 quotes



In [ ]:
# Full pipeline: JS page → Selenium → BS4 → DataFrame
url = "https://quotes.toscrape.com/js/"

# 1. driver.get(url)
driver.get(url)

# 2. WebDriverWait for '.quote' elements
WebDriverWait(driver, 10).until(
    EC.presence_of_all_elements_located((By.CSS_SELECTOR, '.quote')
))

# 3. BeautifulSoup(driver.page_source, ...)
soup = BeautifulSoup(driver.page_source, 'html.parser')

# 4. Extract text, author, tags into list of dicts
quotes_data = []
for quote in soup.find_all(class_='quote'):
  quote_dict = {
      'text': quote.find(class_='text').get_text(strip=True),
      'author': quote.find(class_='author').get_text(strip=True),
      'tags': ', '.join(t.get_text(strip=True) for t in quote.find_all(class_='tag'))
  }
  quotes_data.append(quote_dict)

# 5. pd.DataFrame(...)
df_quotes = pd.DataFrame(quotes_data)
print()
df_quotes.head(10)

,text,author,tags
0,“The world as we have created it is a process ...,Albert Einstein,"change, deep-thoughts, thinking, world"
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling,"abilities, choices"
2,“There are only two ways to live your life. On...,Albert Einstein,"inspirational, life, live, miracle, miracles"
3,"“The person, be it gentleman or lady, who has ...",Jane Austen,"aliteracy, books, classic, humor"
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe,"be-yourself, inspirational"
5,“Try not to become a man of success. Rather be...,Albert Einstein,"adulthood, success, value"
6,“It is better to be hated for what you are tha...,André Gide,"life, love"
7,"“I have not failed. I've just found 10,000 way...",Thomas A. Edison,"edison, failure, inspirational, paraphrased"
8,“A woman is like a tea bag; you never know how...,Eleanor Roosevelt,misattributed-eleanor-roosevelt
9,"“A day without sunshine is like, you know, nig...",Steve Martin,"humor, obvious, simile"


In [ ]:
# Quick analysis!
# 1. value_counts() on 'author' column

print(df_quotes['author'].value_counts())

# 2. Split 'tags' and count individual tags


author
Albert Einstein      3
J.K. Rowling         1
Jane Austen          1
Marilyn Monroe       1
André Gide           1
Thomas A. Edison     1
Eleanor Roosevelt    1
Steve Martin         1
Name: count, dtype: int64


---

## BREAK 2 (10 minutes)

---

---

# Part 5: Exercises — Scraping Dynamic Websites

---

---

## Practice: Exercise 1 — Scraping Page 2 (8 minutes)

Scrape the **second page** of the JavaScript quotes site.

| Detail | Value |
|--------|-------|
| **URL** | `https://quotes.toscrape.com/js/page/2/` |
| **CSS classes** | `.quote` (container), `.text`, `.author`, `.tag` |
| **Expected** | 10 quotes — Marilyn Monroe, J.K. Rowling, Albert Einstein, Bob Marley, Dr. Seuss... |

**Steps:** Selenium → WebDriverWait → BeautifulSoup → extract text, author, tags → print

In [ ]:
# Your code here
driver.get("https://quotes.toscrape.com/js/page/2/")
WebDriverWait(driver, 10).until(
    EC.presence_of_all_elements_located((By.CSS_SELECTOR, '.quote')
))
soup = BeautifulSoup(driver.page_source, 'html.parser')

for quote in soup.find_all(class_='quote'):
  print(f"Text: {quote.find(class_='text').get_text(strip=True)[:30]}")
  print(f"  - {quote.find(class_='author').get_text(strip=True)}")
  print(f"  Tags: {', '.join(t.get_text(strip=True) for t in quote.find_all(class_='tag'))}")
  print()

Text: “This life is what you make it
  - Marilyn Monroe
  Tags: friends, heartbreak, inspirational, life, love, sisters

Text: “It takes a great deal of brav
  - J.K. Rowling
  Tags: courage, friends

Text: “If you can't explain it to a 
  - Albert Einstein
  Tags: simplicity, understand

Text: “You may not be her first, her
  - Bob Marley
  Tags: love

Text: “I like nonsense, it wakes up 
  - Dr. Seuss
  Tags: fantasy

Text: “I may not have gone where I i
  - Douglas Adams
  Tags: life, navigation

Text: “The opposite of love is not h
  - Elie Wiesel
  Tags: activism, apathy, hate, indifference, inspirational, love, opposite, philosophy

Text: “It is not a lack of love, but
  - Friedrich Nietzsche
  Tags: friendship, lack-of-friendship, lack-of-love, love, marriage, unhappy-marriage

Text: “Good friends, good books, and
  - Mark Twain
  Tags: books, contentment, friends, friendship, life

Text: “Life is what happens to us wh
  - Allen Saunders
  Tags: fate, life, misattributed-john-le

---

## Practice: Exercise 2 — Scraping Pages 1–3 into a DataFrame (12 minutes)

| Detail | Value |
|--------|-------|
| **URLs** | `https://quotes.toscrape.com/js/page/1/` through `/page/3/` |
| **Expected** | 30 quotes total (10 per page) |
| **Dict keys** | `text`, `author`, `tags`, `page` |

**Steps:** Loop pages 1–3 → Selenium + wait each → extract → `time.sleep(1)` between pages → DataFrame

In [ ]:
# Your code here
all_quotes = []

for page_num in range(1, 4):
  url = f"https://quotes.toscrape.com/js/page/{page_num}/"
  print(f"Scraping page {page_num}")
  driver.get(url)
  WebDriverWait(driver, 10).until(
    EC.presence_of_all_elements_located((By.CSS_SELECTOR, '.quote')
))
  soup = BeautifulSoup(driver.page_source, 'html.parser')

  for quote in soup.find_all(class_='quote'):
    quote_dict = {
      'text': quote.find(class_='text').get_text(strip=True),
      'author': quote.find(class_='author').get_text(strip=True),
      'tags': ', '.join(t.get_text(strip=True) for t in quote.find_all(class_='tag')),
      'page': page_num
    }
    all_quotes.append(quote_dict)

  time.sleep(1)


df_all = pd.DataFrame(all_quotes)
df_all.head()

Scraping page 1
Scraping page 2
Scraping page 3


,text,author,tags,page
0,“The world as we have created it is a process ...,Albert Einstein,"change, deep-thoughts, thinking, world",1
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling,"abilities, choices",1
2,“There are only two ways to live your life. On...,Albert Einstein,"inspirational, life, live, miracle, miracles",1
3,"“The person, be it gentleman or lady, who has ...",Jane Austen,"aliteracy, books, classic, humor",1
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe,"be-yourself, inspirational",1


In [ ]:
print(f"Total: {len(df_all)} quotes     |      Shape: {df_all.shape}")

Total: 30 quotes     |      Shape: (30, 4)


---

## Practice: Exercise 3 — Analyze the Scraped Data (10 minutes)

Using `df_all` from Exercise 2:

1. How many **unique authors**? Print their names.
2. Which author has the **most quotes**? (`value_counts()`)
3. **Top 5 tags** — split the tags column, count individual tags
4. **Average tags per quote**
5. Print all quotes by **Albert Einstein**

In [ ]:
# Your code here

# 1. How many unique authors?
unique_authors = df_all['author'].unique()
print(unique_authors)
print(f"Number of unique authors: {len(unique_authors)}")

# 2. Which author has the most quotes?
print(df_all['author'].value_counts().head())

# 3. Top 5 tags
all_tags = []
for tag in df_all['tags']:
  all_tags.extend([x.strip() for x in tag.split(',')])

all_tags_series = pd.Series(all_tags)
all_tags_series.value_counts().head()

# 4. Average tags per quote
df_all['n_tags'] = df_all['tags'].apply(lambda x: len(x.split(',')) if x else 0)
print(f"Average tags per quote: {df_all['n_tags'].mean():.1f}")

# 5. Print all quotes by Albert Einstein
albert_quotes = df_all[df_all['author'] == 'Albert Einstein']
for albert_quote in albert_quotes['text']:
  print()
  print(albert_quote[:50])
  print()

['Albert Einstein' 'J.K. Rowling' 'Jane Austen' 'Marilyn Monroe'
 'André Gide' 'Thomas A. Edison' 'Eleanor Roosevelt' 'Steve Martin'
 'Bob Marley' 'Dr. Seuss' 'Douglas Adams' 'Elie Wiesel'
 'Friedrich Nietzsche' 'Mark Twain' 'Allen Saunders' 'Pablo Neruda'
 'Ralph Waldo Emerson' 'Mother Teresa' 'Garrison Keillor' 'Jim Henson']
Number of unique authors: 20
author
Albert Einstein    6
J.K. Rowling       3
Marilyn Monroe     2
Dr. Seuss          2
Bob Marley         2
Name: count, dtype: int64
Average tags per quote: 2.7

“The world as we have created it is a process of o


“There are only two ways to live your life. One is


“Try not to become a man of success. Rather become


“If you can't explain it to a six year old, you do


“If you want your children to be intelligent, read


“Logic will get you from A to Z; imagination will 



---

# Part 6: Daily Challenge — Scraping a New JavaScript Site

---

## Daily Challenge: Scrape an Infinite-Scroll Page

| Detail | Value |
|--------|-------|
| **URL** | `https://quotes.toscrape.com/scroll` |
| **How it works** | JavaScript calls `/api/quotes?page=N` as you scroll down |
| **Total quotes** | ~100 (10 API pages × 10 quotes each) |
| **CSS classes** | Same as before: `.quote`, `.text`, `.author`, `.tag` |

### Steps

**Step 1: Load and scroll** — use this code:
```python
driver.get("https://quotes.toscrape.com/scroll")
for i in range(10):
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(2)
```

**Step 2: Parse** — BeautifulSoup on `driver.page_source`, extract text/author/tags

**Step 3: DataFrame** — should have ~100 rows

**Step 4: Analyze** — unique authors, most quoted, top 10 tags, longest/shortest quote

**Step 5: Save** — `df.to_csv('quotes_scroll.csv', index=False)`

In [ ]:
# Your Daily Challenge code here


### Always close the browser when done!

In [ ]:
driver.quit()
print("Browser closed!")

## Day 2 Summary

### The Pattern
```python
driver = webdriver.Chrome(options=options)  # setup once

driver.get(url)                             # load page (runs JS!)
WebDriverWait(driver, 10).until(            # wait for content
    EC.presence_of_all_elements_located((By.CSS_SELECTOR, '.class')))
soup = BeautifulSoup(driver.page_source, 'html.parser')  # parse (same as Day 1!)

driver.quit()                               # always clean up
```

### When to Use What
| Situation | Tool |
|-----------|------|
| Static HTML | `requests` + BeautifulSoup |
| JavaScript page | Selenium + BeautifulSoup |
| Not sure? | Try `requests` first |

### Key Concepts
| Concept | Meaning |
|---------|---------|
| `driver.get(url)` | Load page in headless Chrome (runs JS!) |
| `driver.page_source` | HTML after JS ran (replaces `response.text`) |
| `WebDriverWait` | Smart wait — continues when element appears |
| `driver.execute_script()` | Run JS commands (e.g., scroll page) |
| `driver.quit()` | Always close browser when done |
| Playwright | Modern Selenium alternative (same concepts) |

```
Static:   URL → requests → BS4 → extract → DataFrame
Dynamic:  URL → Selenium → BS4 → extract → DataFrame
                             ↑
                   Same parsing code!
```